In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -U transformers datasets

In [ ]:
from pathlib import Path
import pandas as pd

path = Path('/kaggle/input/nlp-getting-started')

df = pd.read_csv(path / 'train.csv')
eval_df = pd.read_csv(path / 'test.csv')

df['input'] = df['text']
eval_df['input'] = eval_df['text']

df.head()

In [ ]:
from transformers import AutoTokenizer

model_nm = 'microsoft/deberta-v3-small'
tokz = AutoTokenizer.from_pretrained(model_nm)

def tok_func(x):
    return tokz(x["input"], padding="max_length", truncation=True)

In [ ]:
from datasets import Dataset

ds = Dataset.from_pandas(df[['input', 'target']])
tok_ds = ds.map(tok_func, batched=True)

tok_ds = tok_ds.rename_column('target', 'labels')
tok_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

tok_ds[0]

In [ ]:
dds = tok_ds.train_test_split(test_size=0.2, seed=42)

In [ ]:
eval_ds = Dataset.from_pandas(eval_df[['id', 'input']])
eval_ds = eval_ds.map(tok_func, batched=True)
eval_ds.set_format(type='torch', columns=['input_ids', 'attention_mask'])

eval_ds[0]

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

In [ ]:
from transformers import TrainingArguments

bs = 128
epochs = 1

args = TrainingArguments(
    output_dir='outputs',
    learning_rate=5e-5,
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs,
    weight_decay=0.01,
    report_to='none'
)

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer

model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=2)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dds['train'],
    eval_dataset=dds['test'],
    tokenizer=tokz,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
preds = trainer.predict(eval_ds).predictions
pred_labels = np.argmax(preds, axis=1)

In [ ]:
submission = pd.DataFrame({
    'id': eval_df['id'],
    'target': pred_labels
})

submission.to_csv('submission.csv', index=False)